# Workspace Renaming and Management

This notebook helps with workspace renaming and general management tasks.

## What this notebook does:
- Lists all workspaces with filtering
- Renames workspaces
- Manages workspace users and permissions
- Creates new workspaces
- Bulk operations support

In [ ]:
# Import required modules
import sys
sys.path.append('..')

from modules.fabric_auth import FabricAuth, load_credentials
from modules.fabric_client import FabricClient, workspaces_to_dataframe
from modules.utils import filter_workspaces_by_name, validate_workspace_name, export_to_excel
import pandas as pd

In [ ]:
# Load credentials and authenticate
credentials = load_credentials('../config/credentials.json')
auth = FabricAuth(
    tenant_id=credentials['tenant_id'],
    client_id=credentials['client_id'],
    client_secret=credentials['client_secret']
)

token = auth.get_access_token()
if token:
    print("✓ Authentication successful")
    client = FabricClient(token)
else:
    print("✗ Authentication failed")

In [ ]:
# Get all workspaces
print("Fetching workspaces...")
workspaces = client.get_workspaces()

if workspaces:
    print(f"✓ Found {len(workspaces)} workspaces")
    workspaces_df = workspaces_to_dataframe(workspaces)
    display(workspaces_df[['id', 'name', 'type', 'state']].head(20))
else:
    print("✗ No workspaces found or error occurred")

In [ ]:
# Filter workspaces by name pattern
search_pattern = "test"  # Modify to search for specific workspaces

if workspaces:
    filtered = filter_workspaces_by_name(workspaces, search_pattern)
    print(f"\nWorkspaces matching '{search_pattern}': {len(filtered)}")
    
    if filtered:
        filtered_df = workspaces_to_dataframe(filtered)
        display(filtered_df[['id', 'name', 'type']])

In [ ]:
# Rename a workspace (example)
# Uncomment and modify to rename a workspace

# workspace_id = "YOUR_WORKSPACE_ID"
# new_name = "New Workspace Name"

# # Validate name first
# if validate_workspace_name(new_name):
#     result = client.update_workspace(workspace_id, new_name)
#     if result:
#         print(f"✓ Workspace renamed to '{new_name}'")
#     else:
#         print(f"✗ Failed to rename workspace")
# else:
#     print(f"✗ Invalid workspace name: {new_name}")

In [ ]:
# Bulk rename workspaces with pattern
# Example: Add prefix/suffix to workspaces

def bulk_rename_with_prefix(workspace_list, prefix, dry_run=True):
    """
    Add prefix to workspace names.
    
    Args:
        workspace_list: List of workspace dictionaries
        prefix: Prefix to add
        dry_run: If True, only show what would be renamed
    """
    results = []
    
    for ws in workspace_list:
        old_name = ws['name']
        new_name = f"{prefix}{old_name}"
        
        if validate_workspace_name(new_name):
            if dry_run:
                print(f"Would rename: '{old_name}' -> '{new_name}'")
                results.append({'id': ws['id'], 'old_name': old_name, 'new_name': new_name, 'status': 'planned'})
            else:
                result = client.update_workspace(ws['id'], new_name)
                if result:
                    print(f"✓ Renamed: '{old_name}' -> '{new_name}'")
                    results.append({'id': ws['id'], 'old_name': old_name, 'new_name': new_name, 'status': 'success'})
                else:
                    print(f"✗ Failed to rename: '{old_name}'")
                    results.append({'id': ws['id'], 'old_name': old_name, 'new_name': new_name, 'status': 'failed'})
        else:
            print(f"✗ Invalid new name: '{new_name}'")
            results.append({'id': ws['id'], 'old_name': old_name, 'new_name': new_name, 'status': 'invalid'})
    
    return results

# Example usage (dry run):
# prefix = "PROD_"
# workspaces_to_rename = filter_workspaces_by_name(workspaces, "specific_pattern")
# results = bulk_rename_with_prefix(workspaces_to_rename, prefix, dry_run=True)

In [ ]:
# Create a new workspace
# Uncomment to create a new workspace

# new_workspace_name = "New Workspace Name"

# if validate_workspace_name(new_workspace_name):
#     result = client.create_workspace(new_workspace_name)
#     if result:
#         print(f"✓ Workspace '{new_workspace_name}' created successfully")
#         print(f"Workspace ID: {result.get('id', 'N/A')}")
#     else:
#         print(f"✗ Failed to create workspace")
# else:
#     print(f"✗ Invalid workspace name")

In [ ]:
# Manage workspace users (example)
# Get users for a specific workspace

# workspace_id = "YOUR_WORKSPACE_ID"
# users = client.get_workspace_users(workspace_id)

# if users:
#     print(f"\nUsers in workspace:")
#     users_df = pd.DataFrame(users)
#     display(users_df)

In [ ]:
# Add user to workspace (example)
# Uncomment to add a user

# workspace_id = "YOUR_WORKSPACE_ID"
# user_email = "user@example.com"
# access_right = "Member"  # Options: Admin, Member, Contributor, Viewer

# result = client.add_workspace_user(workspace_id, user_email, access_right)
# if result:
#     print(f"✓ User {user_email} added as {access_right}")
# else:
#     print(f"✗ Failed to add user")

In [ ]:
# Export workspace inventory
if workspaces:
    from datetime import datetime
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    output_filename = f"workspace_inventory_{timestamp}.xlsx"
    
    export_to_excel(workspaces_df, output_filename, sheet_name='Workspaces')
    print(f"\n✓ Workspace inventory exported to {output_filename}")